In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import shapiro
from scipy.stats import ranksums

In [ ]:
task1_df = pd.read_csv("../data/task1_preprocessed.csv")
task2_df = pd.read_csv("../data/task2_preprocessed.csv")
os.makedirs("figures", exist_ok=True)

In [ ]:
task1_ai_group = task1_df[task1_df['ai'] == True]
task1_manual_group = task1_df[task1_df['ai'] == False]
task2_treatment = task2_df[task2_df['treatment'] == 'ai']
task2_control = task2_df[task2_df['treatment'] == 'human']

In [ ]:
# Define color scheme
colors = {
    'ai_dark_gray': '#4F4F4F',
    'manual_work': 'yellow',
    'treatment_blue': '#1f77b4',  # Blue for treatment
    'control_orange': '#ff7f0e'  # Orange for control
}

# Test runs

In [ ]:
# Task 1 descriptive statistics for test runs by AI group
task1_tc_stats = (
    task1_df.groupby('ai')['test_runs']
    .describe()[['count', 'mean', 'std', 'min', '50%', 'max']]
    .rename(columns={'count': 'N', '50%': 'median'})
)

# Task 2 descriptive statistics for test runs by treatment group
task2_tc_stats = (
    task2_df.groupby('treatment')['test_runs']
    .describe()[['count', 'mean', 'std', 'min', '50%', 'max']]
    .rename(columns={'count': 'N', '50%': 'median'})
)

# Display results
print("Task 1 test runs Descriptive Statistics:")
print(task1_tc_stats)
print("\nTask 2 test runs Descriptive Statistics:")
print(task2_tc_stats)

# Task 1

In [ ]:
plt.figure(figsize=(6, 4))

# Drop NaNs and count
ai_test_runs = task1_ai_group['test_runs'].dropna()
manual_test_runs = task1_manual_group['test_runs'].dropna()

# Legend labels with sample sizes
ai_label = f"AI-assisted (N={len(ai_test_runs)})"
manual_label = f"Manual (N={len(manual_test_runs)})"

# Determine combined min/max for consistent binning
combined_min = min(ai_test_runs.min(), manual_test_runs.min())
combined_max = max(ai_test_runs.max(), manual_test_runs.max())
bins = np.linspace(combined_min, combined_max, 50)

# Plot histograms
plt.hist(ai_test_runs, bins=bins, 
         color=colors['ai_dark_gray'], alpha=0.5, edgecolor='black', label=ai_label)
plt.hist(manual_test_runs, bins=bins, 
         color=colors['manual_work'], alpha=0.5, edgecolor='black', label=manual_label)

# Labels and style
plt.xlabel('Test Runs')
plt.ylabel('Frequency')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()

plt.savefig("figures/task1_test_runs_distribution.png", dpi=300)
plt.show()

# Task 2

In [ ]:
plt.figure(figsize=(6, 4))

# Drop NaNs and count
treatment_test_runs = task2_treatment['test_runs'].dropna()
control_test_runs = task2_control['test_runs'].dropna()

# Legend labels with sample sizes
treatment_label = f"AI-assisted (N={len(treatment_test_runs)})"
control_label = f"Manual (N={len(control_test_runs)})"

# Determine combined min/max for consistent binning
combined_min = min(treatment_test_runs.min(), control_test_runs.min())
combined_max = max(treatment_test_runs.max(), control_test_runs.max())
bins = np.linspace(combined_min, combined_max, 50)

# Plot histograms
plt.hist(treatment_test_runs, bins=bins, 
         color=colors['treatment_blue'], alpha=0.5, edgecolor='black', label=treatment_label)
plt.hist(control_test_runs, bins=bins, 
         color=colors['control_orange'], alpha=0.5, edgecolor='black', label=control_label)

# Labels and style
plt.xlabel('Test Runs')
plt.ylabel('Frequency')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()

plt.savefig("figures/task2_test_runs_distribution.png", dpi=300)
plt.show()

## Inferential statistics

In [ ]:
treatment = task2_treatment['test_runs'].dropna()
control = task2_control['test_runs'].dropna()

# Step 1: Perform the Shapiro-Wilk test for normality
shapiro_treatment = shapiro(treatment)
shapiro_control = shapiro(control)

# Step 2: Display the results
print("Shapiro-Wilk Test for Normality:")
print(f"Treatment Group: W={shapiro_treatment.statistic:.4f}, p-value={shapiro_treatment.pvalue:.4e}")
print(f"Control Group: W={shapiro_control.statistic:.4f}, p-value={shapiro_control.pvalue:.4e}")

We reject the null hypothesis of normality.

In [ ]:
# Step 2: Wilcoxon Rank-Sum Test (non-parametric test)
statistic, p_value = ranksums(treatment, control)
print(f"Wilcoxon Rank-Sum Test: Statistic={statistic:.4f}, p-value={p_value:.4e}")

We do not reject the null hypothesis of no difference.

In [ ]:
# Function to compute Cliff's delta
def cliffs_delta(x, y):
    """Compute Cliff's delta for effect size."""
    x, y = x.dropna(), y.dropna()
    n_x, n_y = len(x), len(y)
    more = sum(i > j for i in x for j in y)
    less = sum(i < j for i in x for j in y)
    delta = (more - less) / (n_x * n_y)
    return delta

# Compute Cliff's delta for Test Runs
delta_test_runs = cliffs_delta(task2_treatment['test_runs'], task2_control['test_runs'])

# Print result
print(f"Cliff's Delta for Test Runs: {delta_test_runs:.4f}")

# Interpret effect size magnitude
abs_delta = abs(delta_test_runs)
if abs_delta < 0.147:
    magnitude = "Negligible"
elif abs_delta < 0.33:
    magnitude = "Small"
elif abs_delta < 0.474:
    magnitude = "Medium"
else:
    magnitude = "Large"

print(f"Effect Size Interpretation: {magnitude}")

In [ ]:
def bootstrap_ci_median(data, n_bootstraps=1000000, ci=95):
    """Compute percentile bootstrap confidence interval for the median"""
    boot_medians = [np.median(np.random.choice(data, size=len(data), replace=True)) for _ in range(n_bootstraps)]
    lower = np.percentile(boot_medians, (100 - ci) / 2)
    upper = np.percentile(boot_medians, 100 - (100 - ci) / 2)
    return np.median(data), lower, upper

# Compute Bootstrap CIs for median Test Runs in Task 2
treatment_median, treatment_lower, treatment_upper = bootstrap_ci_median(task2_treatment['test_runs'].dropna())
control_median, control_lower, control_upper = bootstrap_ci_median(task2_control['test_runs'].dropna())

# Error bars
lower_bounds = [treatment_median - treatment_lower, control_median - control_lower]
upper_bounds = [treatment_upper - treatment_median, control_upper - control_median]
yerr = [lower_bounds, upper_bounds]

# Plot
plt.figure(figsize=(5, 5))
x_pos = [0, 1]
medians = [treatment_median, control_median]

plt.errorbar(x_pos, medians, yerr=yerr, fmt='o', capsize=5, capthick=2, color='black')
plt.xticks(x_pos, ['Treatment (AI-assisted)', 'Control (Manual)'])
plt.xlim(-0.5, 1.5)
plt.ylabel('Median Test Runs')
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig("figures/task2_bootstrap_test_runs_ci.png", dpi=300)
plt.show()

# Print summary
print("Bootstrap Confidence Intervals (95% CI) for Median Test Runs:")
print(f"Treatment (AI-assisted): {treatment_median:.2f}  (95% CI: {treatment_lower:.2f} - {treatment_upper:.2f})")
print(f"Control (Manual):       {control_median:.2f}  (95% CI: {control_lower:.2f} - {control_upper:.2f})")

In [ ]:
def bootstrap_ci_median_difference(treatment, control, n_bootstraps=1000000, ci=95):
    """
    Bootstrap percentile CI for the difference in medians (treatment - control).
    """
    treatment = np.asarray(treatment)
    control = np.asarray(control)

    n_t = len(treatment)
    n_c = len(control)
    diffs = np.empty(n_bootstraps)

    for i in range(n_bootstraps):
        t_samp = np.random.choice(treatment, size=n_t, replace=True)
        c_samp = np.random.choice(control, size=n_c, replace=True)
        diffs[i] = np.median(t_samp) - np.median(c_samp)

    lower = np.percentile(diffs, (100 - ci) / 2)
    upper = np.percentile(diffs, 100 - (100 - ci) / 2)
    point = np.median(treatment) - np.median(control)
    return point, lower, upper

# Extract test_runs values
test_runs_treatment = task2_treatment["test_runs"].dropna().to_numpy()
test_runs_control   = task2_control["test_runs"].dropna().to_numpy()

# Compute bootstrap CI for the median difference
tr_diff, tr_lower, tr_upper = bootstrap_ci_median_difference(
    test_runs_treatment, test_runs_control
)

print("Bootstrap CI for the median Test Runs difference (Treatment - Control):")
print(f"Median difference: {tr_diff:.3f}")
print(f"95% CI: [{tr_lower:.3f}, {tr_upper:.3f}]")